# Feature Selection

## Objective

- The goal of this notebook is to identify and select the most relevant features
for churn prediction after feature engineering.

- This notebook is **analysis-only** and does not mutate data used in production.
All final decisions are implemented separately in `src/features/feature_selection.py`.


## Load Feature-Engineered Dataset

In [87]:
import sys
from pathlib import Path

# Add project root to PYTHONPATH
project_root = Path().resolve().parent
sys.path.append(str(project_root))

from src.utils.config import PROCESSED_DATA_PATH

In [88]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.feature_selection import VarianceThreshold
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

plt.style.use("default")

In [89]:
df = pd.read_csv(PROCESSED_DATA_PATH / "telco_customer_churn_feature_engineered.csv")
df.tenure_bin = df.tenure_bin.astype("category").cat.codes


## Dataset Overview

In [90]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,...,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,Churn,tenure_bin,num_active_services,is_month_to_month,is_auto_payment
0,7590-VHVEG,0,0,1,0,1,0,1,29.85,29.85,...,False,False,False,True,False,0,0,1,1,0
1,5575-GNVDE,1,0,0,0,34,1,0,56.95,1889.50,...,True,False,False,False,True,0,2,2,0,0
2,3668-QPYBK,1,0,0,0,2,1,1,53.85,108.15,...,False,False,False,False,True,1,0,2,1,0
3,7795-CFOCW,1,0,0,0,45,0,0,42.30,1840.75,...,True,False,False,False,False,0,2,3,0,1
4,9237-HQITU,0,0,0,0,2,1,1,70.70,151.65,...,False,False,False,True,False,1,0,0,1,0


In [91]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 36 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   customerID                             7043 non-null   object 
 1   gender                                 7043 non-null   int64  
 2   SeniorCitizen                          7043 non-null   int64  
 3   Partner                                7043 non-null   int64  
 4   Dependents                             7043 non-null   int64  
 5   tenure                                 7043 non-null   int64  
 6   PhoneService                           7043 non-null   int64  
 7   PaperlessBilling                       7043 non-null   int64  
 8   MonthlyCharges                         7043 non-null   float64
 9   TotalCharges                           7043 non-null   float64
 10  MultipleLines_No phone service         7043 non-null   bool   
 11  Mult

In [92]:
X = df.drop(columns=["Churn", "customerID"])
y = df["Churn"]

## Low-Variance Feature Analysis

In [93]:
selector = VarianceThreshold(threshold=0.01)
selector.fit(X)

low_variance_features = X.columns[~selector.get_support()]
low_variance_features

Index([], dtype='object')

### **Observations**

- No features with near-zero variance were identified.
- All features exhibit sufficient variability across observations.
- Low-variance filtering did not provide candidates for feature removal at this stage.


## Correlation-Based Feature Filtering

In [78]:
corr_matrix = X.corr().abs()
upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

In [79]:
high_corr_features = [
    col for col in upper.columns
    if any(upper[col] > 0.9)
]
high_corr_features

['MultipleLines_No phone service',
 'OnlineSecurity_No internet service',
 'OnlineBackup_No internet service',
 'DeviceProtection_No internet service',
 'TechSupport_No internet service',
 'StreamingTV_No internet service',
 'StreamingMovies_No internet service']

### **Observations**

- High correlations were observed among features representing
  "No internet service" and "No phone service" states.
- These variables are fully determined by base service subscription features
  and provide redundant information.
- Engineered features did not exhibit extreme linear correlation with their
  source variables, indicating that they capture higher-level behavioral
  patterns rather than direct duplication.


In [80]:
X.drop(columns=high_corr_features, inplace=True)

## Feature–Target Relationship Analysis

### Mutual Information Analysis

In [ ]:
mi_scores = mutual_info_classif(X, y, random_state=42)
mi_scores = pd.Series(mi_scores, index=X.columns).sort_values(ascending=False)
mi_scores

is_month_to_month                        0.098714
tenure                                   0.072427
tenure_bin                               0.064947
Contract_Two year                        0.058185
InternetService_Fiber optic              0.045434
MonthlyCharges                           0.044083
PaymentMethod_Electronic check           0.043326
TotalCharges                             0.043180
InternetService_No                       0.041727
num_active_services                      0.027574
is_auto_payment                          0.022168
OnlineSecurity_Yes                       0.021834
Contract_One year                        0.020672
TechSupport_Yes                          0.018930
Dependents                               0.012425
PaymentMethod_Credit card (automatic)    0.011648
PaperlessBilling                         0.011033
DeviceProtection_Yes                     0.010472
Partner                                  0.008994
OnlineBackup_Yes                         0.007830


### Logistic Regression Feature Importance Analysis
* This section uses Logistic Regression as an analytical tool to assess
conditional feature importance after feature engineering and filtering.

* The model is used strictly for analysis purposes and does not represent
final model training or baseline evaluation, which are handled in later stages
of the project.


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
lr_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)
lr_model.fit(X_scaled, y)

# Extract absolute coefficient values as feature importance
lr_importance = pd.Series(
    np.abs(lr_model.coef_[0]),
    index=X.columns
).sort_values(ascending=False)

lr_importance


tenure                                   1.126636
InternetService_Fiber optic              0.626980
MonthlyCharges                           0.607751
InternetService_No                       0.540034
TotalCharges                             0.489755
Contract_Two year                        0.380278
is_month_to_month                        0.333757
StreamingMovies_Yes                      0.206121
StreamingTV_Yes                          0.199776
MultipleLines_Yes                        0.161686
tenure_bin                               0.157244
PaperlessBilling                         0.153821
OnlineSecurity_Yes                       0.134791
TechSupport_Yes                          0.116035
PaymentMethod_Electronic check           0.108166
SeniorCitizen                            0.081574
PaymentMethod_Mailed check               0.069792
Dependents                               0.069493
is_auto_payment                          0.043895
PhoneService                             0.042158


### **Observations**

Feature removal decisions are based on a combination of Mutual Information
ranking, Logistic Regression coefficient magnitude, and representational
redundancy with stronger or more interpretable features.


**1. Demographic and basic service indicators with negligible predictive signal**

The following features exhibit zero or near-zero Mutual Information and very
low Logistic Regression coefficients, indicating no meaningful association
with churn:

* gender
* Partner
* PhoneService


**2. Individual service subscription indicators with limited incremental value**

These features show weak standalone importance and are largely subsumed by
aggregated behavioral representations, providing minimal additional signal:

* MultipleLines_Yes
* StreamingTV_Yes
* StreamingMovies_Yes
* OnlineBackup_Yes
* DeviceProtection_Yes


**3. Redundant representations of contract commitment**

Contract duration dummy variables are removed in favor of the more
interpretable `is_month_to_month` indicator, which captures customer
commitment and churn risk more directly:

* Contract_One year
* Contract_Two year


**4. Redundant tenure representations**

The binned tenure feature is excluded to avoid duplication with the raw
`tenure` variable, which demonstrates stronger and more consistent importance
across both Mutual Information and Logistic Regression analyses:

* tenure_bin


**5. Derived billing features with high redundancy**

The following feature is removed due to strong dependency on other retained
billing and tenure variables, offering limited incremental explanatory value:

* TotalCharges


**6. Payment method features with inferior or negligible explanatory value**

Granular payment method categories with weak statistical importance are
removed, while behavioral abstractions are preferred where applicable:

* PaymentMethod_Credit card (automatic)
* PaymentMethod_Mailed check


Final feature selection prioritizes interpretability, reduced redundancy, and
consistency across multiple analytical criteria rather than absolute
thresholding on a single metric.

## **Feature Selection Summary**
### **Selected Features**
- tenure
- MonthlyCharges
- InternetService_Fiber optic
- InternetService_No
- is_month_to_month
- num_active_services
- is_auto_payment
- OnlineSecurity_Yes
- TechSupport_Yes
- Dependents
- PaperlessBilling
- SeniorCitizen
- PaymentMethod_Electronic check
